In [9]:
!pip install -U diffusers transformers accelerate torch torchvision -q
!pip uninstall -y torchaudio -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 50.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 77.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.6/554.6 MB 3.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.1/553.1 MB 2.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 8.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 8.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 31.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 923.7 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 94.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 20.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━

In [1]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
login(token=HF_TOKEN)
print("تم تسجيل الدخول ✅")

تم تسجيل الدخول ✅


In [2]:
import requests
url = "https://raw.githubusercontent.com/maram-elaian/brandora/main/data/test_briefs.json"
briefs = requests.get(url).json()
print(f"عدد البريفات: {len(briefs)}")

عدد البريفات: 30


In [3]:
!pip install -U bitsandbytes -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.1 MB/s eta 0:00:00:00:0100:01


In [4]:
import torch, gc
from diffusers import FluxPipeline, FluxTransformer2DModel, BitsAndBytesConfig as DiffusersBnBConfig
from transformers import T5EncoderModel, BitsAndBytesConfig as TransformersBnBConfig

model_id = "black-forest-labs/FLUX.1-schnell"

# تنظيف الذاكرة
gc.collect()
torch.cuda.empty_cache()

print("⏳ جاري تحميل الـ Transformer (4-bit)...")
transformer_quant_config = DiffusersBnBConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
transformer = FluxTransformer2DModel.from_pretrained(
    model_id, 
    subfolder="transformer",
    quantization_config=transformer_quant_config,
    torch_dtype=torch.bfloat16,
)
# نترك الـ transformer في الـ CPU مؤقتاً حتى لا ينفد الذاكرة مع الـ T5
# سننقله للـ GPU فقط عند تشغيل الـ pipeline أو باستخدام cpu_offload

print(f"✅ Transformer loaded | GPU Alloc: {torch.cuda.memory_allocated()/1e9:.2f} GB")

print("\n⏳ جاري تحميل الـ T5 Text Encoder (4-bit) في الـ CPU...")
text_encoder_quant_config = TransformersBnBConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# المفتاح السحري هنا: device_map=None و low_cpu_mem_usage=True
# هذا يمنع transformers من محاولة وضع الأوزان على GPU أثناء التحميل
text_encoder_2 = T5EncoderModel.from_pretrained(
    model_id, 
    subfolder="text_encoder_2",
    quantization_config=text_encoder_quant_config,
    torch_dtype=torch.bfloat16,
    device_map=None,  # <-- مهم جداً: لا تضعه على GPU الآن
    low_cpu_mem_usage=True, # <-- يقلل استهلاك الذاكرة أثناء التحميل
)

print(f"✅ T5 Loaded in CPU | GPU Alloc: {torch.cuda.memory_allocated()/1e9:.2f} GB")

print("\n⏳ تجميع الـ Pipeline...")
pipe = FluxPipeline.from_pretrained(
    model_id,
    transformer=transformer,
    text_encoder_2=text_encoder_2,
    torch_dtype=torch.bfloat16,
    device_map=None, # نتحكم نحن بالنقل
)

# تفعيل CPU Offload ينقل الأجزاء اللازمة للـ GPU لحظة الاستخدام ويعيدها للـ CPU بعدها
# هذا يحل مشكلة امتلاء الذاكرة لأن النموذج كاملاً لن يكون موجوداً على الـ GPU في نفس اللحظة
pipe.enable_model_cpu_offload()

print("✅ الموديل جاهز بنسخة مكمّمة ومعتمدة على CPU Offload")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


⏳ جاري تحميل الـ Transformer (4-bit)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json:   0%|          | 0.00/321 [00:00<?, ?B/s]

(…)ion_pytorch_model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/782 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 80.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 34.81 MiB is free. Including non-PyTorch memory, this process has 14.53 GiB memory in use. Of the allocated memory 14.36 GiB is allocated by PyTorch, and 50.69 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
def brief_to_image_prompt(b):
    return (
        f"{b['logo_direction']}, professional corporate branding, "
        f"flat vector design, Adobe Illustrator style, geometric, "
        f"minimalist icon, clean background, no text, no cartoon, no mascot"
    )
test_briefs = briefs[:30]

for brief in test_briefs:
    prompt = brief_to_image_prompt(brief)
    image = pipe(prompt, num_inference_steps=4, guidance_scale=0.0).images[0]
    filename = f"FLUX_{brief['id']}.png"
    image.save(filename)
    print(f"✅ {brief['id']} → {filename}")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for ax, brief in zip(axes, test_briefs):
    img = Image.open(f"FLUX_{brief['id']}.png")
    ax.imshow(img)
    ax.set_title(brief['id'])
    ax.axis('off')

plt.tight_layout()
plt.show()